# Guided Timing Experiment - Counting Primes

In this activity, you'll measure how long it takes to count the prime numbers below 2,000,000 on Perlmutter. By changing the number of requested CPUs and comparing sequential and parallel execution, you'll investigate how resource allocation affects application performance.

You'll perform five runs, using the first as your baseline for comparison.

### Before you begin

- Run one cell at a time using **Shift+Enter**.
- Commands beginning with `!` execute terminal commands and display their output directly in the notebook.
- Replace `mXXXX` with your NERSC project account whenever submitting a job.

**Do not use Run All.** Each time you execute an `sbatch` cell, you submit a new job to Slurm. Some steps require you to wait for a job to finish before continuing.

## How the files work together

This folder contains four files that you'll use throughout the experiment. The Python programs perform the calculations, while the Slurm scripts request computing resources and launch the programs.

| File | Purpose |
|---|---|
| [prime-counting_serial.py](prime-counting_serial.py) | Counts primes using a single process. |
| [prime-counting_parallel.py](prime-counting_parallel.py) | Divides the work among multiple worker processes. |
| [run_serial.slurm](run_serial.slurm) | Runs the sequential program on Perlmutter. |
| [run_parallel.slurm](run_parallel.slurm) | Runs the parallel program on Perlmutter. |

When you submit a Slurm script using `sbatch`, Slurm schedules your job. Once the requested resources are available and the job starts, the script launches the Python program on a compute node.

Slurm makes the number of requested CPUs available to the batch script through an environment variable called `SLURM_CPUS_PER_TASK`. This allows a program to be configured based on the resources requested.

However, requesting additional CPUs does not automatically improve performance. A program must be designed to take advantage of those resources.

**You do not need to write or modify these files to complete the activity.** If you're curious about how the programs work, you can open the files using the links above.

## Experiment 1: Measure the sequential baseline

We'll begin by running the sequential program with one CPU. This run will serve as our baseline for comparing the performance of subsequent runs.

Submit the sequential Slurm script using the command below. Remember to replace `mXXXX` with your NERSC project account.

- `-A` specifies the project account to charge for the job.
- `--cpus-per-task=1` requests one logical CPU for the task.

In [ ]:
!sbatch -A mXXXX --cpus-per-task=1 run_serial.slurm

When Slurm accepts your submission, it returns a unique **job ID**. You can use this ID to check the status of your job.

Replace `12345678` in the following command with your job ID, then run the cell.

Your job may report one of the following states:

| State | Meaning |
|---|---|
| `PENDING` | Waiting for computing resources or other scheduling requirements. |
| `RUNNING` | Currently executing. |
| `COMPLETED` | Finished successfully. |
| `FAILED` | Encountered an error. |

If no information appears, or your job is still pending or running, wait a moment and run this cell again.

In [ ]:
!sacct -j 12345678 --format=JobID,State,ExitCode

Once your job reports `COMPLETED` with an exit code of `0:0`, you're ready to examine its output.

The Slurm script writes the program's output to a file containing the job ID in its filename. Replace `12345678` with your job ID and run the following command to display the file's contents.

If your job fails, you can also examine this file for error messages, if the file was created.

In [ ]:
!cat prime-serial-12345678.out

Examine the program's output and locate the following measurements:

| Measurement | Description |
|---|---|
| **Workers** | The number of worker processes used by the program. |
| **Prime count** | The total number of primes found below 2,000,000. |
| **Runtime** | The time taken by the program to perform the computation, in seconds. |

The reported runtime will serve as your sequential baseline. It does not include time spent waiting in the Slurm queue.

Keep these results visible in your notebook. You'll use them to compare performance at the end of the activity.

## Experiment 2: Change the CPU request

Now let's run the same sequential program, but this time request eight CPUs instead of one. We'll keep the program and problem size the same so that the only change we introduce is the number of requested CPUs.

**Before submitting the job, consider:** How do you expect the runtime to change compared to your baseline?

Remember to replace `mXXXX` with your NERSC project account.

In [ ]:
!sbatch -A mXXXX --cpus-per-task=8 run_serial.slurm

As in the previous experiment, replace `12345678` in both cells below with your new job ID. Check the job's status and wait until it has completed successfully before examining its output.

In [ ]:
!sacct -j 12345678 --format=JobID,State,ExitCode

In [ ]:
!cat prime-serial-12345678.out

Compare the reported **Workers**, **Prime count**, and **Runtime** with those from Experiment 1.

Did the results match your expectations? Consider the relationship between the number of CPUs you requested and the number of workers the program used.

## Experiment 3: Divide the work

Now let's switch to the parallel program. Rather than checking every number sequentially, this program divides the numbers into smaller ranges that multiple worker processes can check simultaneously. The results are then combined to determine the total prime count.

The parallel Slurm script uses `SLURM_CPUS_PER_TASK` to determine how many worker processes the Python program should use.

We'll run the program with two, four, and eight workers to investigate how increasing the number of workers affects runtime.

### Two workers

Let's begin by requesting two CPUs. How do you expect the runtime to compare with your sequential baseline?

Remember to replace `mXXXX` with your NERSC project account.

In [ ]:
!sbatch -A mXXXX --cpus-per-task=2 run_parallel.slurm

Replace `12345678` in both cells below with your new job ID. Wait for the job to complete successfully before examining its output.

In [ ]:
!sacct -j 12345678 --format=JobID,State,ExitCode

In [ ]:
!cat prime-parallel-12345678.out

### Four workers

Now let's double the number of requested CPUs while keeping the program and problem size the same.

**Before submitting the job, consider:** Will doubling the number of workers cut the runtime in half?

In [ ]:
!sbatch -A mXXXX --cpus-per-task=4 run_parallel.slurm

Replace `12345678` in both cells below with your new job ID. Wait for the job to complete successfully before examining its output.

In [ ]:
!sacct -j 12345678 --format=JobID,State,ExitCode

In [ ]:
!cat prime-parallel-12345678.out

### Eight workers

For our final run, we'll double the number of requested CPUs once more.

**Before submitting the job, consider:** Based on your two-worker and four-worker measurements, what runtime would you expect with eight workers?

In [ ]:
!sbatch -A mXXXX --cpus-per-task=8 run_parallel.slurm

Replace `12345678` in both cells below with your new job ID. Wait for the job to complete successfully before examining its output.

In [ ]:
!sacct -j 12345678 --format=JobID,State,ExitCode

In [ ]:
!cat prime-parallel-12345678.out

## Compare your results

You've now completed five runs of the same computational problem using different resource configurations. Let's compare your measurements to see how these configurations affected performance.

Before continuing, verify that all five runs reported the same **Prime count**.

### Record your measurements

Enter the **Runtime** reported by each program in the corresponding variable below. Replace each `None` with your measured runtime in seconds, then run the cell.

These measurements will be used to generate the figures in the following sections.

In [ ]:
# Experiment 1
serial_1 = None

# Experiment 2
serial_8 = None

# Experiment 3
parallel_2 = None
parallel_4 = None
parallel_8 = None

# Check that all measurements have been entered.
times = [serial_1, serial_8, parallel_2, parallel_4, parallel_8]

if any(t is None for t in times):
    print("Please enter all five runtime measurements.")
elif any(not isinstance(t, (int, float)) or t <= 0 for t in times):
    print("Each runtime must be a positive number.")
else:
    print("All five runtime measurements have been recorded.")

### Compare runtimes

The following figure compares the runtime of all five runs. Each bar represents a different resource configuration.

Run the cell below to generate the figure. Remember that a shorter bar represents a faster computation.

As you examine the figure, consider:

- How did requesting additional CPUs affect the sequential program?
- How did increasing the number of workers affect the parallel program?

You do not need to modify the plotting code.

In [ ]:
import matplotlib.pyplot as plt

assert all(
    isinstance(t, (int, float)) and t > 0 for t in times
), "Enter all five positive runtime measurements before continuing."

labels = [
    'Sequential\n1 CPU',
    'Sequential\n8 CPUs',
    'Parallel\n2 CPUs',
    'Parallel\n4 CPUs',
    'Parallel\n8 CPUs'
]

colors = [
    '#64748b',
    '#64748b',
    '#2878b5',
    '#2878b5',
    '#2878b5'
]

fig, ax = plt.subplots(figsize=(10, 5), layout='constrained')

bars = ax.bar(labels, times, color=colors)

ax.bar_label(bars, fmt='%.4f', padding=4)
ax.set_xlabel('Program and requested CPUs')
ax.set_ylabel('Application runtime (seconds)')
ax.set_title('Prime-counting runtime on Perlmutter')
ax.set_ylim(0, max(times) * 1.2)
ax.grid(axis='y', alpha=0.2)
ax.set_axisbelow(True)

plt.show()

### Evaluate parallel speedup

Another way to compare performance is by measuring **speedup**: how much faster a computation runs relative to a baseline.

We calculate speedup by dividing the sequential baseline runtime by the runtime of a parallel run.

$$
\text{Speedup} = \frac{\text{Sequential baseline runtime}}{\text{Parallel runtime}}
$$

For example, if the sequential program takes 20 seconds and the parallel program takes 10 seconds, the speedup is 2. The parallel program completed the computation in half the time.

The following figure compares your measured speedup with an ideal reference, where doubling the number of workers halves the runtime.

Run the cell below to generate the figure. How closely do your measurements follow the ideal reference?

In [ ]:
workers = [2, 4, 8]

parallel_times = [parallel_2, parallel_4, parallel_8]

assert all(
    isinstance(t, (int, float)) and t > 0
    for t in [serial_1, *parallel_times]
), "Enter valid baseline and parallel runtimes before continuing."

speedups = [serial_1 / t for t in parallel_times]

fig, ax = plt.subplots(figsize=(8, 5), layout='constrained')

# Measured parallel speedup
ax.plot(
    workers, speedups,
    'o-',
    color='#2878b5',
    label='Measured parallel speedup'
)

# Sequential baseline
ax.scatter(
    1, 1,
    color='#64748b',
    label='Sequential baseline',
    zorder=3
)

# Ideal speedup reference
ax.plot(
    [1, 2, 4, 8], [1, 2, 4, 8],
    '--',
    color='#64748b',
    label='Ideal speedup'
)

# Label the measured values
for x, y in zip(workers, speedups):
    ax.annotate(
        f'{y:.2f}×',
        (x, y),
        xytext=(0, 9),
        textcoords='offset points',
        ha='center'
    )

ax.set(
    xticks=[1, 2, 4, 8],
    xlabel='Number of workers',
    ylabel='Speedup relative to sequential baseline',
    title='Parallel prime-counting speedup'
)

ax.set_ylim(0, max(8, max(speedups)) * 1.15)
ax.grid(alpha=0.2)
ax.legend()

plt.show()

## Interpret your measurements

Using your measurements and figures, answer the following questions. Double-click this cell to edit it, write your answers beneath each question, and press **Shift+Enter** when finished.

1. How did the two sequential runtimes compare? Use the reported worker counts to explain your observations.

   **Answer:**

2. How did the runtime change as you increased the number of workers in the parallel program? Reference your measurements.

   **Answer:**

3. Did doubling the number of workers halve the runtime each time? How did your measured speedup compare with the ideal reference?

   **Answer:**

4. If you were to run this computation again on Perlmutter, which resource configuration would you choose and why? Consider both application performance and the responsible use of shared computing resources.

   **Answer:**

## Submit your notebook

Before submitting, verify that your notebook contains:

- Output from all five runs
- All five runtime measurements
- Both performance figures
- Answers to all four reflection questions

Save your completed notebook. In the JupyterLab file browser, right-click `prime-counting.ipynb` and select **Download**.

Upload the completed `.ipynb` file to Canvas under **4.3.1 Prime Counting Activity**.